# LitScope — Fine-Tuning Step 1: Generate Synthetic Queries

Uses DeepSeek to generate realistic IS researcher search queries for each behavioral paper.
Output is used as training data for SPECTER fine-tuning in the next notebook.

**Pipeline:**
1. Load `papers_classified.csv` — use `theories_used` column to guide generation
2. For each behavioral paper, generate 3 queries via DeepSeek
3. Quality check: sample 20 queries for human review
4. Save `synthetic_queries.csv`

**Prerequisite:** `02_classify_papers.ipynb` must have been run (need `papers_classified.csv`)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR   = '/content/drive/MyDrive/LitScope/data'
MODELS_DIR  = '/content/drive/MyDrive/LitScope/models'
os.makedirs(MODELS_DIR, exist_ok=True)

CLASSIFIED_CSV     = f'{DRIVE_DIR}/papers_classified.csv'
QUERIES_CSV        = f'{DRIVE_DIR}/synthetic_queries.csv'

print(f'Data dir:   {DRIVE_DIR}')
print(f'Models dir: {MODELS_DIR}')

## 2. Install Dependencies

In [ ]:
!pip install openai pandas tqdm -q

## 3. Configuration

In [ ]:
from google.colab import userdata
from openai import OpenAI

DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url='https://api.deepseek.com',
)

# Queries per paper — 3 gives enough variety without blowing cost
QUERIES_PER_PAPER = 3
AUTOSAVE_EVERY    = 50   # save progress every N papers

print('Configuration complete')
print(f'API key present: {bool(DEEPSEEK_API_KEY)}')

## 4. Load Behavioral Papers

In [ ]:
import pandas as pd

df = pd.read_csv(CLASSIFIED_CSV)
behavioral_df = df[df['is_behavioral'] == True].copy().reset_index(drop=True)

print(f'Total papers:      {len(df)}')
print(f'Behavioral papers: {len(behavioral_df)}  (query generation target)')
print()
print('Sample theories_used values:')
for t in behavioral_df['theories_used'].dropna().head(5):
    print(f'  {t}')

## 5. Generate Synthetic Queries

The prompt uses `theories_used` from DeepSeek classification to steer generation toward IS-specific vocabulary.
Three query styles per paper: broad topic, theory-specific, method/context-specific.

In [ ]:
import json
import time
import os
from tqdm import tqdm

_PROMPT = """You are an expert in Information Systems (IS) research.

Given this IS paper, generate {n} realistic search queries that a researcher would type to find it.

Paper Title: {title}
Abstract (excerpt): {abstract}
Theories Used: {theories}

Requirements:
- Use IS-specific concepts and domain vocabulary (e.g. TAM, UTAUT, TPB, trust, privacy, adoption, intention, continuance, satisfaction, social influence)
- Do NOT include statistical methods (SEM, PLS, regression, survey) unless they are the paper's primary contribution
- Do NOT copy the paper title or reuse the same combination of keywords from the title — rephrase using synonyms or a different conceptual framing
- Generate {n} queries covering different angles:
  1. Broad topic query (research area + context, no theory name)
  2. Theory-specific query (name the theory + application domain)
  3. Application or outcome query (who uses it, in what context, or what the key finding implies)
- Each query: 5–15 words
- Do NOT use the word "paper" or "study" in the query

Return JSON only, no other text:
{{"queries": ["query1", "query2", "query3"]}}"""


def generate_queries(title, abstract, theories):
    prompt = _PROMPT.format(
        n=QUERIES_PER_PAPER,
        title=title,
        abstract=str(abstract)[:400],
        theories=theories if theories and str(theories) != 'nan' else 'Not specified',
    )
    try:
        resp = client.chat.completions.create(
            model='deepseek-chat',
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=250,
            temperature=0.7,
        )
        content = resp.choices[0].message.content.strip()
        content = content.replace('```json', '').replace('```', '').strip()
        parsed = json.loads(content)
        queries = parsed.get('queries', [])
        return [q for q in queries if len(q.split()) >= 5]
    except Exception as e:
        print(f'  [ERROR] {e}')
        return []


# Resume from existing file if interrupted
if os.path.exists(QUERIES_CSV):
    existing = pd.read_csv(QUERIES_CSV)
    done_dois = set(existing['doi'].unique())
    results = existing.to_dict('records')
    print(f'Resuming — {len(done_dois)} papers already done, {len(behavioral_df) - len(done_dois)} remaining')
else:
    done_dois = set()
    results = []
    print(f'Starting fresh — {len(behavioral_df)} papers to process')

pending = behavioral_df[~behavioral_df['doi'].isin(done_dois)]
print(f'Papers to process this run: {len(pending)}\n')

for i, (_, row) in enumerate(tqdm(pending.iterrows(), total=len(pending))):
    queries = generate_queries(row['title'], row['abstract'], row['theories_used'])
    for q in queries:
        results.append({'query': q, 'doi': row['doi'], 'title': row['title']})

    if (i + 1) % AUTOSAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(QUERIES_CSV, index=False, encoding='utf-8-sig')
        print(f'  ── Auto-saved ({i+1}/{len(pending)}) ──')

    time.sleep(0.3)

queries_df = pd.DataFrame(results)
queries_df.to_csv(QUERIES_CSV, index=False, encoding='utf-8-sig')
print(f'\nDone! Generated {len(queries_df)} queries for {queries_df["doi"].nunique()} papers')
print(f'Saved: {QUERIES_CSV}')

## 6. Quality Check — Sample Review

**Core test:** Verify generated queries are:
- IS-domain appropriate (not generic)
- Not copying paper titles
- Covering different angles (broad / theory / method)

Review the 20 samples below manually before proceeding to fine-tuning.

In [ ]:
import random
import re

queries_df = pd.read_csv(QUERIES_CSV)
sample_dois = random.sample(list(queries_df['doi'].unique()), min(7, queries_df['doi'].nunique()))

print('=' * 70)
print('  QUERY QUALITY SAMPLE  (7 papers × 3 queries)')
print('  Check: IS vocab? Not copying title? Varied angles?')
print('=' * 70)

for doi in sample_dois:
    group = queries_df[queries_df['doi'] == doi]
    title = group['title'].iloc[0]
    print(f'\nPaper: {title[:75]}')
    for j, q in enumerate(group['query'].tolist(), 1):
        title_words = set(title.lower().split())
        query_words = set(q.lower().split())
        overlap = len(title_words & query_words) / max(len(query_words), 1)
        flag = '  ⚠️  HIGH TITLE OVERLAP' if overlap > 0.5 else ''
        print(f'  Q{j}: {q}{flag}')

print()
print('─' * 70)

# Automated stats
query_lengths = queries_df['query'].str.split().str.len()
print(f'\nQuery length stats:')
print(f'  Mean:  {query_lengths.mean():.1f} words')
print(f'  Min:   {query_lengths.min()}')
print(f'  Max:   {query_lengths.max()}')
print(f'  <5 words (too short): {(query_lengths < 5).sum()}')
print(f'  >15 words (too long): {(query_lengths > 15).sum()}')

# IS keyword coverage — use word-boundary matching to avoid false positives
IS_TERMS = ['TAM', 'UTAUT', 'TPB', 'TRA', 'SDT', 'SEM', 'PLS', 'survey',
            'adoption', 'behavioral', 'intention', 'trust', 'privacy',
            'mobile', 'e-commerce', 'social', 'technology', 'acceptance']
all_queries_text = ' '.join(queries_df['query'].tolist()).lower()
print(f'\nIS term coverage in generated queries:')
for term in IS_TERMS:
    count = len(re.findall(r'\b' + re.escape(term.lower()) + r'\b', all_queries_text))
    if count > 0:
        print(f'  {term:<15} {count:>4} occurrences')

# Method-term saturation check
METHOD_TERMS = ['sem', 'pls', 'survey', 'regression', 'anova', 'experiment']
print(f'\nMethod-term saturation (should be low):')
for term in METHOD_TERMS:
    count = len(re.findall(r'\b' + term + r'\b', all_queries_text))
    flag = '  ⚠️' if count > 100 else ''
    print(f'  {term:<15} {count:>4} occurrences{flag}')

print(f'\nTotal: {len(queries_df)} queries for {queries_df["doi"].nunique()} papers')